In [ ]:
#| default_exp channel

In [ ]:
#| export
# channel.py

from __future__ import annotations

import random
from abc import ABC, abstractmethod

from comm_core.core import Transmission, TransmissionResult
from comm_core.topology import Link


    

In [ ]:
#| export

class Channel(ABC):

    @abstractmethod
    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:
        pass

    def reset(self, seed: int | None = None) -> None:
        "Start over (e.g. a new episode); `seed` reseeds the channel's random generator, if it has one."
        pass

    def update(self, time: float) -> None:
        pass

In [ ]:
#| export

class IdentityChannel(Channel):
    """
    Perfect communication.
    """

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        message = transmission.message

        return TransmissionResult(
            success=True,
            transmission=transmission,
            payload=message.payload,
            latency=0.0,
            size_bits=message.size_bits or 0,
            snr_db=float("inf"),
            error_rate=0.0,
        )




In [ ]:
#| export

class AWGNChannel(Channel):
    """
    Simple analytical channel.

    This intentionally models only packet success/failure
    rather than individual bits.
    """

    def __init__(
        self,
        snr_db: float, # Signal-to-noise ratio (reported in the results)
        packet_error_rate: float = 0.0, # Probability that a packet is lost
        seed: int | None = None, # Seed of the channel's own random generator
    ):
        self.snr_db = snr_db
        self.packet_error_rate = packet_error_rate
        self.rng = random.Random(seed)

    def reset(self, seed: int | None = None) -> None:
        "`seed` reseeds the packet losses."
        if seed is not None:
            self.rng.seed(seed)

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        success = (
            self.rng.random()
            >= self.packet_error_rate
        )

        message = transmission.message

        return TransmissionResult(
            success=success,
            transmission=transmission,
            payload=(
                message.payload
                if success
                else None
            ),
            latency=0.0,
            size_bits=message.size_bits or 0,
            snr_db=self.snr_db,
            error_rate=self.packet_error_rate,
        )

### Tests

In [ ]:
from comm_core.core import Message, Transmission
from comm_core.topology import Link

def losses(channel, n=200):
    t = Transmission(message=Message('a', 'b', payload=1, size_bits=8), sender='a', receiver='b', start_time=0.0)
    return [channel.transmit(t, Link('a', 'b')).success for _ in range(n)]

assert all(losses(IdentityChannel()))
first = losses(AWGNChannel(snr_db=5, packet_error_rate=0.3, seed=0))
assert first == losses(AWGNChannel(snr_db=5, packet_error_rate=0.3, seed=0))      # seeded
assert 0.2 < 1 - sum(first) / len(first) < 0.4
channel = AWGNChannel(snr_db=5, packet_error_rate=0.3, seed=1)
a = losses(channel)
channel.reset(seed=1)
assert losses(channel) == a                                                         # reset(seed) replays

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()